## Thread vs Process

### Process :
    A process is an independent instance of a program being executed. When you open your web browser, that is a process. When you open a video player, that is a different process.
### Thread: 
    A thread is a "lightweight" unit of execution that lives inside a process. A single process can have many threads.

### GIL (GLOBAL INTERPRETER LOCK)
In most languages (like C++ or Java), threads can run on different CPU cores simultaneously to do heavy math. However, standard Python (CPython) has a Global Interpreter Lock (GIL).The GIL is like a single microphone in the "house." Even if you have five threads (people), only the person holding the microphone is allowed to execute Python code at that moment.Use Threads for I/O-bound tasks (waiting for a website to respond or a file to save). While one thread waits, it drops the "microphone" so another thread can work.Use Processes for CPU-bound tasks (calculating Pi to a billion digits). Each process gets its own Python interpreter and its own "microphone," allowing them to run on separate CPU cores.

In [4]:
# program to spawn 3 threads printing number from 1 to 10
import threading
import time
def print_numbers(thread_name):
    for i in range(10):
        time.sleep(0.01)
        print(f"{thread_name}: {i}")

t1 = threading.Thread(target = print_numbers, args=("Thread-1",))
t2 = threading.Thread(target = print_numbers, args=("Thread-2",))
t3 = threading.Thread(target = print_numbers, args=("Thread-3",))

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()


Thread-1: 0Thread-2: 0
Thread-3: 0

Thread-2: 1
Thread-3: 1
Thread-1: 1
Thread-3: 2
Thread-1: 2
Thread-2: 2
Thread-1: 3
Thread-3: 3
Thread-2: 3
Thread-3: 4
Thread-2: 4
Thread-1: 4
Thread-3: 5
Thread-1: 5
Thread-2: 5
Thread-3: 6
Thread-1: 6
Thread-2: 6
Thread-3: 7
Thread-1: 7
Thread-2: 7
Thread-3: 8
Thread-2: 8
Thread-1: 8
Thread-2: 9
Thread-3: 9
Thread-1: 9


## Race Condition
A race condition occurs when the timing or order of events affects the correctness of a program. In multi-threaded programming, this usually happens when two or more threads access shared data and try to change it at the same time.

Because the threads are "racing" to finish their task, the final state of the data depends on which thread wins the race, often leading to unpredictable bugs.

## Threading Lock
A Lock (or Mutex) acts as a "talking stick." Only the thread holding the lock is allowed to modify the protected data. Others must wait in line.

In [10]:
import threading
database = 0
def increase():
    global database 
    for _ in range(100000):
        temp = database
        temp+=1
        time.sleep(0)
        database = temp
    
print(f"Starting Value: {database}")
thread1 = threading.Thread(target = increase)
thread2 = threading.Thread(target = increase)

thread1.start()
thread2.start()

thread1.join()
thread2.join()

print(f"End Value: {database}")

Starting Value: 0
End Value: 100000


In [11]:
import threading
database = 0
thread_lock = threading.Lock()
def increase():
    global database 
    for _ in range(100000):
        with thread_lock:
            temp = database
            temp+=1
            time.sleep(0)
            database = temp
    
print(f"Starting Value: {database}")
thread1 = threading.Thread(target = increase)
thread2 = threading.Thread(target = increase)

thread1.start()
thread2.start()

thread1.join()
thread2.join()

print(f"End Value: {database}")

Starting Value: 0
End Value: 200000


## What is a Deadlock?
A deadlock happens when two or more threads are permanently stuck waiting for each other to release resources.

Analogy: Person A has the paper but needs the pen. Person B has the pen but needs the paper. Neither will yield, so nothing gets signed.

### How it Happens (Circular Wait)
Thread A acquires Lock 1, then waits for Lock 2.

Thread B acquires Lock 2, then waits for Lock 1.

Both threads are now blocked infinitely.

### The Fix: Lock Ordering
To prevent this, enforce a global rule: All threads must acquire locks in the exact same order.

The New Flow: You mandate that every thread must ask for Lock 1 before Lock 2.

The Result: Thread A grabs Lock 1. Thread B tries to grab Lock 1 but is forced to wait. Thread A can freely grab Lock 2, finish its work, and release both locks. Thread B wakes up and safely proceeds. No collisions!

### Q- Implement a thread-safe stack using `threading.Lock()` and Test it with 4 threads doing push/pop concurrently

In [11]:
import threading
import time
import random
class ThreadSafeStack:
    def __init__(self):
        self.stack = []
        self.lock = threading.Lock()
    
    def push(self, val):
        with self.lock:
            self.stack.append(val)
    
    def pop(self):
        with self.lock:
            if not self.stack:
                return None
            return self.stack.pop
    
    def size(self):
        with self.lock:
            return len(self.stack)

In [12]:
def worker(stack, thread_id):
    for i in range(50):
        val = f"T{thread_id}-{i}"
        stack.push(val)
        # Slight sleep to encourage thread switching/contention
        time.sleep(random.uniform(0.001, 0.01))
    
    for _ in range(50):
        stack.pop()
        time.sleep(random.uniform(0.001, 0.01))

def run_test():
    my_stack = ThreadSafeStack()
    threads = []
    
    print("Starting 4 threads...")
    for i in range(4):
        t = threading.Thread(target=worker, args=(my_stack, i))
        threads.append(t)
        t.start()

    # Wait for all threads to finish
    for t in threads:
        t.join()

    print(f"Final Stack Size: {my_stack.size()}")
    if my_stack.size() == 0:
        print("Success: Stack is empty after equal pushes and pops!")
    else:
        print("Failure: Stack size is inconsistent!")

if __name__ == "__main__":
    run_test()

Starting 4 threads...
Final Stack Size: 200
Failure: Stack size is inconsistent!


### IMplementing thread-safe LRU Cache

In [14]:
import threading
class Node:
    def __init__(self, key, val):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}
        self.lock = threading.RLock()

        self.head = Node(0,0)
        self.tail = Node(0,0)
        self.head.next = self.tail
        self.tail.prev = self.head

    def remove(self, node):
        prv, nxt = node.prev, node.next
        prv.next = nxt
        next.prev = prv
    
    def add_to_front(self, node):
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key: int) -> int:
        with self.lock:
            if key not in self.cache:
                return -1
            node = self.cache[key]
            self.remove(node)
            self.add_to_front(node)
            return node.val
        
    def put(self, key, value):
        with self.lock:
            if key in self.cache:
                self.remove(self.cache[key])
            node = Node(key, value)
            self.cache[key] = node
            self.add_to_front(node)
            if len(self.cache) > self.capacity:
                lru = self.tail.prev
                self.remove(lru)
                del self.cache[lru.key]


### Concurrency: threading.Condition() — wait/notify
When to use it?
Use Condition when threads must wait for a state change (not just mutual exclusion).
Example: consumers wait until buffer has items; producers wait until buffer has space.
Core contract - 
- Condition wraps a lock.
- You must hold that lock when calling wait(), notify(), notify_all().
- wait() atomically:
1) releases the lock,
2) sleeps,
3) re-acquires lock before returning.
Always wait in a while loop (while not predicate: cond.wait()) to handle spurious wakeups / races.

In [15]:
import threading
import time
import random
from collections import deque

In [22]:
class BoundedBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.q = deque()
        self.cond = threading.Condition()
    
    def put(self, item):
        with self.cond:
            while len(self.q)>=self.capacity:
                self.cond.wait() # wait until consumer remvoes something
            self.q.append(item)
            print(f"[PUT ] {item}, size={len(self.q)}")
            self.cond.notify()  # wake one waiter (likely a consumer)

    def get(self):
        with self.cond:
          while not self.q:
              self.cond.wait()  # wait until producer adds something
          item = self.q.popleft()
          print(f"[GET ] {item}, size={len(self.q)}")
          self.cond.notify()  # wake one waiter (likely a producer)
          return item
    
def producer(buf: BoundedBuffer, n=10):
    for i in range(n):
        time.sleep(random.uniform(0.05, 0.2))
        buf.put(f"item-{i}")

def consumer(buf: BoundedBuffer, n=10):
    for _ in range(10):
        time.sleep(random.uniform(0.1, 0.25))
        _ = buf.get()

buf = BoundedBuffer(capacity = 10)
t1 = threading.Thread(target=producer, args=(buf, 12))
t3 = threading.Thread(target=producer, args=(buf, 12))
t5 = threading.Thread(target=producer, args=(buf, 12))
t2 = threading.Thread(target=consumer, args=(buf, 12))
t4 = threading.Thread(target=consumer, args=(buf, 12))

t1.start()
t2.start()
t3.start()
t4.start()
t5.start()
t1.join()
t2.join()
t3.join()
t4.join()
t5.join()
print("Done")


[PUT ] item-0, size=1
[PUT ] item-0, size=2
[GET ] item-0, size=1
[PUT ] item-0, size=2
[GET ] item-0, size=1
[PUT ] item-1, size=2
[PUT ] item-1, size=3
[PUT ] item-1, size=4
[PUT ] item-2, size=5
[GET ] item-0, size=4
[PUT ] item-2, size=5
[PUT ] item-2, size=6
[PUT ] item-3, size=7
[PUT ] item-4, size=8
[PUT ] item-3, size=9
[GET ] item-1, size=8
[PUT ] item-3, size=9
[GET ] item-1, size=8
[GET ] item-1, size=7
[PUT ] item-4, size=8
[PUT ] item-4, size=9
[PUT ] item-5, size=10
[GET ] item-2, size=9
[PUT ] item-5, size=10
[GET ] item-2, size=9
[PUT ] item-5, size=10
[GET ] item-2, size=9
[PUT ] item-6, size=10
[GET ] item-3, size=9
[PUT ] item-6, size=10
[GET ] item-4, size=9
[PUT ] item-6, size=10
[GET ] item-3, size=9
[PUT ] item-7, size=10
[GET ] item-3, size=9
[PUT ] item-7, size=10
[GET ] item-4, size=9
[PUT ] item-8, size=10
[GET ] item-4, size=9
[PUT ] item-7, size=10
[GET ] item-5, size=9
[PUT ] item-8, size=10
[GET ] item-5, size=9
[PUT ] item-8, size=10
[GET ] item-5, size=

KeyboardInterrupt: 

## Reader Writer Problem

### Here are the strict rules we must enforce:

Multiple readers can read from the shared resource simultaneously (reading doesn't change the data, so it's safe).

Only one writer can write to the shared resource at a time.

No readers can read while a writer is writing (to prevent reading half-written or corrupt data).
To solve this in Python, we need to carefully manage access using synchronization primitives like threading.Lock, threading.RLock, or threading.Semaphore.

The Strategy
To allow multiple concurrent readers but exclusive writers, we need three things:

readers_count: An integer to keep track of how many readers are currently reading.

read_lock (Mutex): A lock to protect the readers_count variable so multiple threads don't scramble the count.

write_lock (Semaphore or Lock): A lock to protect the actual shared resource.

The Magic Trick: * The first reader to arrive acquires the write_lock, effectively locking out all writers.

Subsequent readers can just join in without checking the write_lock.

The last reader to leave releases the write_lock, signaling that writers can now safely access the resource.

In [28]:
import threading
import time
import random

class ReaderWriter:
    def __init__(self):
        self.shared_data =0

        self.reader_count = 0 
        self.read_lock = threading.Lock()
        self.write_lock = threading.Lock()

    def read(self, reader_id):
        with self.read_lock:
            self.reader_count += 1
            if self.reader_count == 1:
                self.write_lock.acquire()
        
        print(f"[Reader {reader_id}] Started reading. Data = {self.shared_data}. (Total readers: {self.reader_count})")
        time.sleep(random.uniform(0.1, 0.5)) # Simulate reading time
        print(f"[Reader {reader_id}] Finished reading.")

        with self.read_lock:
            self.reader_count -= 1
            if self.reader_count == 0:
                self.write_lock.release()
            
    def write(self, writer_id):
        self.write_lock.acquire()
        try:
            print(f"  -> [Writer {writer_id}] Started writing. Locking everyone out.")
            time.sleep(random.uniform(0.5, 1.0)) # Simulate writing time
            self.shared_data += 1
            print(f"  -> [Writer {writer_id}] Finished writing. New Data = {self.shared_data}.")
        finally:
            # Release the write lock
            self.write_lock.release()



In [29]:
rw = ReaderWriter()
threads = [] 
for i in range(5):
    t_read = threading.Thread(target = rw.read, args= (i+1,))
    threads.append(t_read)

    if i%2 == 0:
        t_write = threading.Thread(target = rw.write, args=(i+1,))
        threads.append(t_write)

for t in threads:
    t.start()
    time.sleep(0.05)

for t in threads:
    t.join()

print("Simulation complete")

[Reader 1] Started reading. Data = 0. (Total readers: 1)
[Reader 2] Started reading. Data = 0. (Total readers: 2)
[Reader 3] Started reading. Data = 0. (Total readers: 3)
[Reader 1] Finished reading.
[Reader 4] Started reading. Data = 0. (Total readers: 3)
[Reader 5] Started reading. Data = 0. (Total readers: 4)
[Reader 2] Finished reading.
[Reader 3] Finished reading.
[Reader 4] Finished reading.
[Reader 5] Finished reading.
  -> [Writer 1] Started writing. Locking everyone out.
  -> [Writer 1] Finished writing. New Data = 1.
  -> [Writer 3] Started writing. Locking everyone out.
  -> [Writer 3] Finished writing. New Data = 2.
  -> [Writer 5] Started writing. Locking everyone out.
  -> [Writer 5] Finished writing. New Data = 3.
Simulation complete


## IMplementing Parallel BFS

In [48]:
import concurrent.futures
import threading
import time

def parallel_bfs(graph, start_node):
    # Track the shortest distance to each node and protect it with a lock
    distances = {start_node: 0}
    distances_lock = threading.Lock()
    
    # The current level of nodes we are exploring
    frontier = [start_node]
    current_level = 0
    
    # Initialize the thread pool
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        
        while frontier:
            print(f"Level {current_level} (Nodes: {len(frontier)}): {frontier}")
            
            next_frontier = []
            next_frontier_lock = threading.Lock()

            # The worker function executed by the threads
            def explore_neighbors(node):
                local_next_frontier = []
                
                # Look at all neighbors of the assigned node
                for neighbor in graph.get(node, []):
                    # Safely check and update the global visited/distances state
                    with distances_lock:
                        if neighbor not in distances:
                            distances[neighbor] = current_level + 1
                            local_next_frontier.append(neighbor)
                
                # Safely merge the locally discovered nodes into the global next frontier
                if local_next_frontier:
                    with next_frontier_lock:
                        next_frontier.extend(local_next_frontier)

            # executor.map assigns the explore_neighbors function to every node in the frontier.
            # Wrapping it in list() blocks execution until ALL threads finish the current level.
            # This is our crucial Level-Synchronization barrier!
            list(executor.map(explore_neighbors, frontier))
            
            # Move to the next level
            frontier = next_frontier
            current_level += 1

    return distances

In [49]:
sample_graph = {
        'A': ['B', 'C', 'D'],
        'B': ['A', 'E', 'F'],
        'C': ['A', 'G'],
        'D': ['A', 'H'],
        'E': ['B', 'I'],
        'F': ['B'],
        'G': ['C', 'J', 'K'],
        'H': ['D'],
        'I': ['E'],
        'J': ['G'],
        'K': ['G']
    }

print("Starting Parallel BFS...")
start_time = time.time()

result_distances = parallel_bfs(sample_graph, 'A')

print("\nTraversal Complete!")
print(f"Distances from 'A': {result_distances}")

Starting Parallel BFS...
Level 0 (Nodes: 1): ['A']
Level 1 (Nodes: 3): ['B', 'C', 'D']
Level 2 (Nodes: 4): ['E', 'F', 'G', 'H']
Level 3 (Nodes: 3): ['I', 'J', 'K']

Traversal Complete!
Distances from 'A': {'A': 0, 'B': 1, 'C': 1, 'D': 1, 'E': 2, 'F': 2, 'G': 2, 'H': 2, 'I': 3, 'J': 3, 'K': 3}


## Dining Philosopher Problem

The Dining Philosophers problem is a classic computer science thought experiment that perfectly illustrates the dangers of poorly managed concurrency. It visualizes a scenario where five philosophers sit around a table with only five forks between them, yet each requires two forks to eat. This setup mimics software systems where multiple distinct processes (the philosophers) must share limited, mutually exclusive resources (the forks) to complete their tasks. If the system lacks strict rules for how these resources are acquired, it inevitably faces deadlock—where every process grabs one resource, waits forever for a second one, and the whole system freezes—or starvation, where a process is perpetually denied the resources it needs to ever execute

In [58]:
import threading
import time
import random

class Philosopher(threading.Thread):
    def __init__(self, index, forks):
        super().__init__(name=f"Philosopher-{index}")
        self.index = index
        
        # Determine the indices of the required forks
        left_fork_index = index
        right_fork_index = (index + 1) % len(forks)
        
        # THE FIX: Resource Hierarchy
        # Break the symmetry by forcing everyone to grab the lower-indexed lock first.
        # For Philosopher 4, left is 4, right is 0. They will grab 0 first, then 4.
        self.first_fork = forks[min(left_fork_index, right_fork_index)]
        self.second_fork = forks[max(left_fork_index, right_fork_index)]

    def run(self):
        # Each philosopher will think and eat a couple of times
        for _ in range(2): 
            self.think()
            self.eat()

    def think(self):
        print(f"[{self.name}] is deeply in thought...")
        time.sleep(random.uniform(0.1, 0.3))

    def eat(self):
        print(f"[{self.name}] is getting hungry.")
        
        # Using 'with' acts as a context manager, automatically calling 
        # lock.acquire() on entry and lock.release() on exit, even if errors occur.
        with self.first_fork:
            with self.second_fork:
                print(f"[{self.name}] secured both forks and is EATING.")
                time.sleep(random.uniform(0.1, 0.3))
        
        print(f"[{self.name}] finished eating and dropped the forks.")


In [59]:
num_philosophers = 5
table_forks = [threading.Lock() for _ in range(num_philosophers)]
philosophers = [Philosopher(i, table_forks) for i in range(num_philosophers)]

for p in philosophers:
    p.start()

for p in philosophers:
    p.join()

print("DINNER CONCLUDED SUCCESSFULLY. No deadlocks")

[Philosopher-0] is deeply in thought...
[Philosopher-1] is deeply in thought...
[Philosopher-2] is deeply in thought...
[Philosopher-3] is deeply in thought...
[Philosopher-4] is deeply in thought...
[Philosopher-3] is getting hungry.
[Philosopher-3] secured both forks and is EATING.
[Philosopher-4] is getting hungry.
[Philosopher-0] is getting hungry.
[Philosopher-1] is getting hungry.
[Philosopher-1] secured both forks and is EATING.
[Philosopher-2] is getting hungry.
[Philosopher-3] finished eating and dropped the forks.
[Philosopher-3] is deeply in thought...
[Philosopher-4] secured both forks and is EATING.
[Philosopher-1] finished eating and dropped the forks.[Philosopher-2] secured both forks and is EATING.

[Philosopher-1] is deeply in thought...
[Philosopher-3] is getting hungry.
[Philosopher-2] finished eating and dropped the forks.
[Philosopher-2] is deeply in thought...
[Philosopher-1] is getting hungry.
[Philosopher-1] secured both forks and is EATING.
[Philosopher-4] fini

## Thread-Safe Priority Queue using `threading.Condition`

### Why not just use a Lock?
A `Lock` gives mutual exclusion — only one thread touches the heap at a time.  
But what happens when a consumer calls `get()` on an **empty** queue?

| Approach | Problem |
|----------|---------|
| Return `None` immediately | Caller must busy-loop / poll — wastes CPU |
| `time.sleep()` then retry | Adds latency + still polling |
| **`Condition.wait()`** | Thread *sleeps* until a producer calls `notify()` — **zero CPU waste, instant wake-up** |

### How `Condition` solves it
```
consumer                          producer
─────────                         ────────
cond.acquire()                    cond.acquire()
while heap is empty:              heappush(item)
    cond.wait()   ← sleeps        cond.notify()  ← wakes consumer
item = heappop()                  cond.release()
cond.release()
```
- `wait()` atomically **releases the lock + sleeps**. When notified it **re-acquires the lock** before returning.
- Always use `while` (not `if`) to guard `wait()` — protects against **spurious wakeups** and races where another consumer steals the item first.

### Priority Queue Recap
- Backed by a **min-heap** (`heapq`): smallest value = highest priority.
- We store `(priority, seq, data)` tuples. The `seq` counter breaks ties in FIFO order so items with equal priority come out in insertion order.

In [60]:
import heapq
import threading
import time
import random

class ThreadSafePriorityQueue:
    def __init__(self):
        self._heap = []
        self._seq = 0                        # tie-breaker for equal priorities
        self._cond = threading.Condition()    # wraps its own RLock internally

    def put(self, item, priority: int = 0):
        """Add an item. Lower priority number = higher urgency."""
        with self._cond:
            heapq.heappush(self._heap, (priority, self._seq, item))
            self._seq += 1
            self._cond.notify()              # wake ONE waiting consumer

    def get(self, timeout=None):
        """Remove and return the highest-priority item.
        Blocks until an item is available (or until timeout seconds elapse).
        Raises TimeoutError if timeout expires.
        """
        with self._cond:
            # Always use a while-loop — guards against spurious wakeups
            # and the race where another consumer grabbed the item first.
            while not self._heap:
                got_notified = self._cond.wait(timeout=timeout)
                if not got_notified:         # timed out
                    raise TimeoutError("get() timed out waiting for an item")
            priority, _, item = heapq.heappop(self._heap)
            return priority, item

    def peek(self):
        """Return the highest-priority item without removing it."""
        with self._cond:
            if not self._heap:
                return None
            priority, _, item = self._heap[0]
            return priority, item

    def __len__(self):
        with self._cond:
            return len(self._heap)

print("ThreadSafePriorityQueue class defined ✓")

ThreadSafePriorityQueue class defined ✓


### Demo 1 — Basic priority ordering
Push tasks with different priorities, then drain. Lowest number comes out first.

In [62]:
pq = ThreadSafePriorityQueue()

pq.put("low-priority-email",  priority=5)
pq.put("CRITICAL-alert",      priority=1)
pq.put("medium-log",          priority=3)
pq.put("CRITICAL-backup",     priority=1)   # same priority as alert — FIFO within tier
pq.put("debug-trace",         priority=10)

print("Draining in priority order:")
while len(pq):
    pri, item = pq.get()
    print(f"  priority={pri}  →  {item}")

Draining in priority order:
  priority=1  →  CRITICAL-alert
  priority=1  →  CRITICAL-backup
  priority=3  →  medium-log
  priority=5  →  low-priority-email
  priority=10  →  debug-trace


### Demo 2 — Multi-threaded: Producers + Consumers with blocking `get()`
- **3 producers** each push 5 tasks with random priorities.  
- **2 consumers** call `get()` in a loop — when the queue is empty they **block** (no busy-looping!) until a producer pushes something.  
- After all producers finish we push **poison pills** (`None`) so consumers know to shut down.

In [63]:
POISON = None
NUM_PRODUCERS = 3
NUM_CONSUMERS = 2
ITEMS_PER_PRODUCER = 5

pq = ThreadSafePriorityQueue()
processed = []               # collect results for final verification
processed_lock = threading.Lock()

def producer(pid):
    for i in range(ITEMS_PER_PRODUCER):
        pri = random.randint(1, 10)
        task = f"P{pid}-task{i}"
        pq.put(task, priority=pri)
        print(f"[Producer-{pid}] put {task!r} (pri={pri})")
        time.sleep(random.uniform(0.05, 0.15))
    print(f"[Producer-{pid}] done producing.")

def consumer(cid):
    while True:
        pri, item = pq.get()          # blocks here if queue empty — no CPU waste
        if item is POISON:
            print(f"[Consumer-{cid}] got poison pill, shutting down.")
            break
        print(f"[Consumer-{cid}] processing {item!r} (pri={pri})")
        with processed_lock:
            processed.append((pri, item))
        time.sleep(random.uniform(0.05, 0.1))

# --- Launch threads ---
threads = []
for i in range(NUM_PRODUCERS):
    t = threading.Thread(target=producer, args=(i,))
    threads.append(t)
    t.start()

consumer_threads = []
for i in range(NUM_CONSUMERS):
    t = threading.Thread(target=consumer, args=(i,))
    consumer_threads.append(t)
    t.start()

# Wait for all producers to finish
for t in threads:
    t.join()

# Send one poison pill per consumer so each one exits cleanly
for _ in range(NUM_CONSUMERS):
    pq.put(POISON, priority=999)

for t in consumer_threads:
    t.join()

print(f"\n=== All done. Processed {len(processed)} items ===")
print("Items in order they were consumed:")
for pri, item in processed:
    print(f"  pri={pri}  {item}")

[Producer-0] put 'P0-task0' (pri=6)[Producer-1] put 'P1-task0' (pri=1)

[Producer-2] put 'P2-task0' (pri=3)
[Consumer-0] processing 'P1-task0' (pri=1)
[Consumer-1] processing 'P2-task0' (pri=3)
[Producer-1] put 'P1-task1' (pri=4)
[Consumer-0] processing 'P1-task1' (pri=4)
[Consumer-1] processing 'P0-task0' (pri=6)
[Producer-2] put 'P2-task1' (pri=7)
[Producer-0] put 'P0-task1' (pri=5)
[Producer-1] put 'P1-task2' (pri=9)
[Consumer-1] processing 'P0-task1' (pri=5)
[Consumer-0] processing 'P2-task1' (pri=7)
[Producer-2] put 'P2-task2' (pri=9)
[Consumer-0] processing 'P1-task2' (pri=9)
[Consumer-1] processing 'P2-task2' (pri=9)
[Producer-0] put 'P0-task2' (pri=10)
[Producer-1] put 'P1-task3' (pri=4)
[Consumer-0] processing 'P1-task3' (pri=4)
[Producer-0] put 'P0-task3' (pri=1)
[Producer-2] put 'P2-task3' (pri=9)
[Consumer-1] processing 'P0-task3' (pri=1)
[Producer-1] put 'P1-task4' (pri=5)
[Consumer-0] processing 'P1-task4' (pri=5)
[Producer-0] put 'P0-task4' (pri=1)
[Consumer-1] processin

### Demo 3 — `get()` with timeout
Shows what happens when a consumer tries to `get()` from an empty queue with a timeout — it raises `TimeoutError` instead of blocking forever.

In [64]:
pq = ThreadSafePriorityQueue()

def delayed_producer():
    time.sleep(0.8)
    pq.put("late-arrival", priority=1)
    print("[Producer] pushed 'late-arrival' after 0.8s delay")

threading.Thread(target=delayed_producer).start()

# Attempt 1: timeout too short — will raise
try:
    print("Consumer: trying get(timeout=0.3) on empty queue...")
    pq.get(timeout=0.3)
except TimeoutError as e:
    print(f"Consumer: caught TimeoutError → {e}")

# Attempt 2: timeout long enough — producer will have pushed by now
print("\nConsumer: trying get(timeout=2.0)...")
pri, item = pq.get(timeout=2.0)
print(f"Consumer: got {item!r} with priority={pri}")

Consumer: trying get(timeout=0.3) on empty queue...
Consumer: caught TimeoutError → get() timed out waiting for an item

Consumer: trying get(timeout=2.0)...
[Producer] pushed 'late-arrival' after 0.8s delayConsumer: got 'late-arrival' with priority=1



### Key Takeaways

| Concept | Detail |
|---------|--------|
| **`Condition` vs `Lock`** | `Lock` = mutual exclusion only. `Condition` = mutual exclusion **+ wait/notify signaling** |
| **`wait()` in a `while` loop** | Guards against spurious wakeups and the race where another thread consumed the item between `notify()` and your wake-up |
| **`notify()` vs `notify_all()`** | `notify()` wakes **one** waiter (efficient). Use `notify_all()` when **all** waiters might need to re-check (e.g., shutdown signal) |
| **Poison pill pattern** | Push a sentinel value to signal consumers to exit — clean shutdown without `daemon` threads |
| **Tie-breaking with `seq`** | `heapq` compares tuples element-by-element. A monotonic counter as the second element ensures FIFO order within the same priority |

## Thread-Safe Trie — Coarse-Grained vs Fine-Grained Locking

A **Trie** (prefix tree) maps strings to values by walking one character per node. Making it thread-safe raises a key design question:

### Coarse-Grained Locking (one lock for the whole trie)
```
lock.acquire()           # every thread waits here
  walk / modify nodes
lock.release()
```
- **Pros:** Simple, correct, zero chance of deadlock.
- **Cons:** All operations are serialized — inserting "apple" blocks a search for "zoo" even though they share zero nodes.

### Fine-Grained Locking (one lock per node)
```
node.lock.acquire()      # only blocks threads touching THIS node
  read/create child
  move to child
node.lock.release()      # release parent before locking child → "hand-over-hand"
```
- **Pros:** Maximum concurrency — threads on different branches never block each other.
- **Cons:** More memory (one lock per node), more complex code, must lock in a consistent order (parent → child) to avoid deadlocks.

### Hand-Over-Hand (Lock Coupling)
The fine-grained version uses **hand-over-hand** locking:
1. Lock the current node.
2. Lock the child node.
3. **Release** the current node.
4. Move to the child, repeat.

This ensures you never hold more than **two locks** at once and always acquire in parent→child order — no deadlocks possible.

### Approach 1 — Coarse-Grained Trie (single `RLock`)

In [ ]:
import threading
import time
import random

class TrieNodeSimple:
    __slots__ = ('children', 'is_end')
    def __init__(self):
        self.children = {}
        self.is_end = False

class CoarseGrainedTrie:
    """One RLock guards the entire trie.
    Simple & correct, but all operations are serialized."""

    def __init__(self):
        self.root = TrieNodeSimple()
        self.lock = threading.RLock()

    def insert(self, word: str):
        with self.lock:
            node = self.root
            for ch in word:
                if ch not in node.children:
                    node.children[ch] = TrieNodeSimple()
                node = node.children[ch]
            node.is_end = True

    def search(self, word: str) -> bool:
        with self.lock:
            node = self.root
            for ch in word:
                if ch not in node.children:
                    return False
                node = node.children[ch]
            return node.is_end

    def starts_with(self, prefix: str) -> bool:
        with self.lock:
            node = self.root
            for ch in prefix:
                if ch not in node.children:
                    return False
                node = node.children[ch]
            return True

print("CoarseGrainedTrie defined ✓")

### Approach 2 — Fine-Grained Trie (hand-over-hand locking per node)

In [ ]:
class TrieNodeFine:
    __slots__ = ('children', 'is_end', 'lock')
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.lock = threading.Lock()      # each node has its own lock

class FineGrainedTrie:
    """Hand-over-hand locking: lock parent → lock child → release parent.
    Threads on disjoint branches never block each other."""

    def __init__(self):
        self.root = TrieNodeFine()

    def insert(self, word: str):
        parent = self.root
        parent.lock.acquire()
        for ch in word:
            if ch not in parent.children:
                parent.children[ch] = TrieNodeFine()
            child = parent.children[ch]
            child.lock.acquire()       # lock child BEFORE releasing parent
            parent.lock.release()
            parent = child
        parent.is_end = True
        parent.lock.release()

    def search(self, word: str) -> bool:
        parent = self.root
        parent.lock.acquire()
        for ch in word:
            if ch not in parent.children:
                parent.lock.release()
                return False
            child = parent.children[ch]
            child.lock.acquire()
            parent.lock.release()
            parent = child
        result = parent.is_end
        parent.lock.release()
        return result

    def starts_with(self, prefix: str) -> bool:
        parent = self.root
        parent.lock.acquire()
        for ch in prefix:
            if ch not in parent.children:
                parent.lock.release()
                return False
            child = parent.children[ch]
            child.lock.acquire()
            parent.lock.release()
            parent = child
        parent.lock.release()
        return True

print("FineGrainedTrie defined ✓")

### Correctness Test — Hammer both tries with 8 threads
4 threads insert words, 4 threads search concurrently. Both tries should produce identical results.

In [ ]:
WORDS = [
    "apple", "app", "application", "apply", "ape",
    "banana", "band", "ban", "bat", "bath",
    "cat", "car", "card", "care", "cart",
    "dog", "dot", "dome", "door", "down",
]

SEARCH_WORDS = WORDS + ["ap", "ba", "xyz", "do", "card", "zoo"]

def stress_test(trie, label):
    errors = []
    errors_lock = threading.Lock()

    def inserter(words):
        for w in words:
            trie.insert(w)
            time.sleep(random.uniform(0.001, 0.005))

    def searcher(words, expected_inserted):
        """Search repeatedly; after all inserters join, every word must be found."""
        time.sleep(0.05)  # let some inserts happen first
        for w in words:
            result = trie.search(w)
            # We only flag an error if the word SHOULD be in the trie
            # (i.e., all inserts are done) but isn't found.
            if w in expected_inserted and not result:
                with errors_lock:
                    errors.append(f"{w} not found but should exist")

    # Split words across 4 inserter threads
    chunks = [WORDS[i::4] for i in range(4)]
    threads = []
    for chunk in chunks:
        t = threading.Thread(target=inserter, args=(chunk,))
        threads.append(t)
        t.start()

    # 4 searcher threads run concurrently with inserters
    for _ in range(4):
        t = threading.Thread(target=searcher, args=(SEARCH_WORDS, set(WORDS)))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    # Final verification: every word must now be findable
    missing = [w for w in WORDS if not trie.search(w)]
    prefix_fails = [w for w in WORDS if not trie.starts_with(w[:2])]

    print(f"[{label}] All {len(WORDS)} words inserted.")
    print(f"[{label}] Missing after all inserts : {missing or 'None ✓'}")
    print(f"[{label}] Prefix check failures     : {prefix_fails or 'None ✓'}")
    print(f"[{label}] Race errors during search  : {errors or 'None ✓'}")
    print()

print("=== Coarse-Grained Trie ===")
stress_test(CoarseGrainedTrie(), "Coarse")

print("=== Fine-Grained Trie ===")
stress_test(FineGrainedTrie(), "Fine")

### Concurrency Comparison — Timing both approaches
Inserts and searches happen simultaneously. The fine-grained trie should show better throughput because threads on different branches (`a*` vs `d*`) don't block each other.

In [ ]:
import string

def generate_random_words(n, max_len=8):
    return [''.join(random.choices(string.ascii_lowercase, k=random.randint(3, max_len)))
            for _ in range(n)]

NUM_WORDS = 2000
NUM_THREADS = 8
words = generate_random_words(NUM_WORDS)
chunks = [words[i::NUM_THREADS] for i in range(NUM_THREADS)]

def bench(trie_cls, label):
    trie = trie_cls()
    start = time.perf_counter()

    threads = []
    for chunk in chunks:
        # Half threads insert, half search (on same words — some may not exist yet)
        t_ins = threading.Thread(target=lambda c: [trie.insert(w) for w in c], args=(chunk,))
        t_srch = threading.Thread(target=lambda c: [trie.search(w) for w in c], args=(chunk,))
        threads.extend([t_ins, t_srch])

    for t in threads:
        t.start()
    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start
    found = sum(1 for w in words if trie.search(w))
    print(f"[{label}] {NUM_WORDS} inserts + {NUM_WORDS} searches | "
          f"time={elapsed:.4f}s | found={found}/{NUM_WORDS}")

bench(CoarseGrainedTrie, "Coarse")
bench(FineGrainedTrie,   "Fine  ")

### Key Takeaways — Coarse vs Fine-Grained Locking

| Aspect | Coarse-Grained | Fine-Grained (Hand-over-Hand) |
|--------|---------------|-------------------------------|
| **Locks** | 1 `RLock` for entire trie | 1 `Lock` per `TrieNode` |
| **Concurrency** | Fully serialized — one thread at a time | Threads on different branches run in parallel |
| **Deadlock risk** | None (single lock) | None if you always lock parent→child (consistent ordering) |
| **Complexity** | Very low | Higher — must carefully acquire/release at every step |
| **Memory** | Minimal | One lock object per node (adds up for large tries) |
| **When to pick** | Low contention, small trie, prototyping | High contention, read-heavy workloads, large tries |

**Interview tip:** Start by saying "I'd use coarse-grained for simplicity," then proactively discuss how you'd upgrade to fine-grained if contention becomes a bottleneck. This shows you understand the trade-off spectrum.

---
## `asyncio` — Cooperative Concurrency (Single-Threaded)

### The Big Idea: Cooperative vs Preemptive
| Model | Who decides when to switch? | Example |
|-------|--------------------------|---------|
| **Threads** | The OS (preemptive — interrupts at any moment) | `threading.Thread` |
| **asyncio** | **You** (cooperative — you explicitly `await` to yield control) | `async def` + `await` |

Because asyncio runs in **one thread**, there is no GIL contention, no race conditions on shared state, and no need for locks — as long as you never do CPU-heavy work inside an `async def` without offloading it.

### The Event Loop
```
Event Loop
│
├── runs a coroutine until it hits an `await`
├── suspends it, picks up another ready coroutine
└── resumes the original when its I/O is done
```

### Core vocabulary

| Keyword / function | What it means |
|--------------------|---------------|
| `async def f()` | Declares a **coroutine function**. Calling `f()` returns a coroutine object — it does NOT run yet. |
| `await expr` | Suspends this coroutine until `expr` completes; yields control back to the event loop. |
| `asyncio.run(coro)` | Creates an event loop, runs `coro` to completion, closes the loop. Entry point. |
| `asyncio.sleep(n)` | Async version of `time.sleep` — yields to the loop instead of blocking the thread. |
| `asyncio.gather(*coros)` | Runs multiple coroutines **concurrently**; returns when all finish. |
| `asyncio.create_task(coro)` | Schedules a coroutine to run concurrently without `await`ing immediately. |

### Example 1 — Your first `async` function
Understand the lifecycle: define → call (gets coroutine) → `await` (actually runs)

In [65]:
import asyncio

# --- Step 1: define a coroutine ---
async def greet(name: str, delay: float):
    print(f"[{name}] starting — will wait {delay}s")
    await asyncio.sleep(delay)        # ← yields to event loop here
    print(f"[{name}] done after {delay}s")
    return f"Hello, {name}!"

# --- Step 2: calling greet() returns a coroutine OBJECT, does NOT run it ---
coro = greet("Alice", 1.0)
print(type(coro))          # <class 'coroutine'>

# --- Step 3: run it with asyncio.run ---
# In Jupyter we need await directly (the notebook already has a running event loop)
result = await greet("Alice", 0.5)
print("Return value:", result)

<class 'coroutine'>
[Alice] starting — will wait 0.5s
[Alice] done after 0.5s
Return value: Hello, Alice!


### Example 2 — Sequential `await` vs `asyncio.gather` (concurrent)
This is the most important pattern to understand intuitively.

**Sequential** — each `await` finishes before the next starts → total = sum of delays.  
**`gather`** — all coroutines start immediately, suspend at their own `await`s → total ≈ max delay.

In [66]:
import time

async def fetch(resource: str, delay: float):
    """Simulates a network fetch that takes `delay` seconds."""
    print(f"  → fetching {resource}...")
    await asyncio.sleep(delay)
    print(f"  ✓ received {resource}")
    return f"data:{resource}"

# ── SEQUENTIAL ──────────────────────────────────────────────────
print("=== Sequential awaits ===")
t0 = time.perf_counter()

r1 = await fetch("users",    0.3)
r2 = await fetch("products", 0.4)
r3 = await fetch("orders",   0.2)

print(f"Sequential total: {time.perf_counter()-t0:.2f}s  (≈ 0.3+0.4+0.2 = 0.9s)\n")

# ── CONCURRENT with gather ───────────────────────────────────────
print("=== asyncio.gather (concurrent) ===")
t0 = time.perf_counter()

results = await asyncio.gather(
    fetch("users",    0.3),
    fetch("products", 0.4),
    fetch("orders",   0.2),
)

print(f"Concurrent total: {time.perf_counter()-t0:.2f}s  (≈ max(0.3,0.4,0.2) = 0.4s)")
print("All results:", results)

=== Sequential awaits ===
  → fetching users...
  ✓ received users
  → fetching products...
  ✓ received products
  → fetching orders...
  ✓ received orders
Sequential total: 0.91s  (≈ 0.3+0.4+0.2 = 0.9s)

=== asyncio.gather (concurrent) ===
  → fetching users...
  → fetching products...
  → fetching orders...
  ✓ received orders
  ✓ received users
  ✓ received products
Concurrent total: 0.41s  (≈ max(0.3,0.4,0.2) = 0.4s)
All results: ['data:users', 'data:products', 'data:orders']


### Example 3 — `create_task`: fire-and-forget concurrency
`create_task` schedules a coroutine immediately without blocking the caller.  
Useful when you want to kick off background work and continue doing something else.

In [68]:
async def background_logger(interval: float, count: int):
    for i in range(count):
        await asyncio.sleep(interval)
        print(f"    [logger] heartbeat #{i+1}")

async def main_work():
    # Kick off background task — it runs concurrently, we don't await it yet
    logger_task = asyncio.create_task(background_logger(0.2, 4))

    print("[main] starting heavy work...")
    for step in range(3):
        await asyncio.sleep(0.3)          # simulates I/O work
        print(f"[main] work step {step+1} complete")

    print("[main] waiting for logger to finish...")
    await logger_task                     # now we wait for it
    print("[main] all done.")

await main_work()

[main] starting heavy work...
    [logger] heartbeat #1
[main] work step 1 complete
    [logger] heartbeat #2
[main] work step 2 complete
    [logger] heartbeat #3
    [logger] heartbeat #4
[main] work step 3 complete
[main] waiting for logger to finish...
[main] all done.


### Example 4 — The classic mistake: blocking the event loop
`time.sleep()` inside an `async def` **blocks the entire thread** — no other coroutine can run.  
Always use `await asyncio.sleep()` inside async code.

In [69]:
import time as _time

# ❌ BAD — time.sleep blocks the thread; B can't start until A finishes
async def bad_task(name, delay):
    print(f"[{name}] start")
    _time.sleep(delay)          # ← BLOCKING: freezes the event loop
    print(f"[{name}] end")

print("=== BAD (blocking sleep) — B waits for A to fully finish ===")
t0 = time.perf_counter()
await asyncio.gather(bad_task("A", 0.4), bad_task("B", 0.4))
print(f"elapsed: {time.perf_counter()-t0:.2f}s  (≈ 0.8s, serialized)\n")

# ✅ GOOD — asyncio.sleep yields; both run concurrently
async def good_task(name, delay):
    print(f"[{name}] start")
    await asyncio.sleep(delay)  # ← NON-BLOCKING: yields to loop
    print(f"[{name}] end")

print("=== GOOD (async sleep) — A and B overlap ===")
t0 = time.perf_counter()
await asyncio.gather(good_task("A", 0.4), good_task("B", 0.4))
print(f"elapsed: {time.perf_counter()-t0:.2f}s  (≈ 0.4s, concurrent)")

=== BAD (blocking sleep) — B waits for A to fully finish ===
[A] start
[A] end
[B] start
[B] end
elapsed: 0.81s  (≈ 0.8s, serialized)

=== GOOD (async sleep) — A and B overlap ===
[A] start
[B] start
[A] end
[B] end
elapsed: 0.40s  (≈ 0.4s, concurrent)


### Example 5 — Offloading CPU-bound work with `run_in_executor`
asyncio is single-threaded. Heavy CPU work (compression, parsing, math) will block the loop just like `time.sleep()`.  
Fix: `loop.run_in_executor(None, fn, *args)` runs the function in a thread pool, **yielding the loop** while the CPU work executes on another OS thread.

In [70]:
def cpu_heavy(n: int) -> int:
    """Pure CPU work — sum of squares up to n."""
    return sum(i * i for i in range(n))

async def process_items(items):
    loop = asyncio.get_event_loop()
    tasks = [
        loop.run_in_executor(None, cpu_heavy, item)   # offload to thread pool
        for item in items
    ]
    results = await asyncio.gather(*tasks)
    return results

t0 = time.perf_counter()
results = await process_items([5_000_000, 4_000_000, 6_000_000])
print(f"CPU-bound results computed concurrently in {time.perf_counter()-t0:.2f}s")
print(f"First result (sum of squares 0..5M): {results[0]:,}")

CPU-bound results computed concurrently in 0.48s
First result (sum of squares 0..5M): 41,666,654,166,667,500,000


---
## Threading vs asyncio — When to Use Which

The decision tree comes down to **one question: what are your tasks mostly waiting on?**

### Head-to-head: same I/O-bound problem solved both ways
We simulate hitting 5 "APIs" (each taking a random delay). Watch the timing difference.

In [71]:
import threading
import time

ENDPOINTS = {
    "users":       0.30,
    "products":    0.40,
    "inventory":   0.25,
    "orders":      0.35,
    "analytics":   0.50,
}

# ─────────────────────────────────────────────
# Approach A: threading
# ─────────────────────────────────────────────
def sync_fetch(name, delay, results, lock):
    time.sleep(delay)                    # blocks this thread, not others
    with lock:
        results[name] = f"data:{name}"

def threading_approach():
    results, lock = {}, threading.Lock()
    threads = [
        threading.Thread(target=sync_fetch, args=(n, d, results, lock))
        for n, d in ENDPOINTS.items()
    ]
    for t in threads: t.start()
    for t in threads: t.join()
    return results

t0 = time.perf_counter()
thread_results = threading_approach()
thread_time = time.perf_counter() - t0
print(f"threading  → {thread_time:.3f}s  (≈ max delay = 0.50s)")

# ─────────────────────────────────────────────
# Approach B: asyncio
# ─────────────────────────────────────────────
async def async_fetch(name, delay):
    await asyncio.sleep(delay)
    return name, f"data:{name}"

async def asyncio_approach():
    pairs = await asyncio.gather(*[async_fetch(n, d) for n, d in ENDPOINTS.items()])
    return dict(pairs)

t0 = time.perf_counter()
async_results = await asyncio_approach()
async_time = time.perf_counter() - t0
print(f"asyncio    → {async_time:.3f}s  (≈ max delay = 0.50s)")

print(f"\nSame results? {set(thread_results) == set(async_results)}")
print(f"Overhead diff: threading used {thread_time - async_time:+.3f}s more than asyncio")

threading  → 0.527s  (≈ max delay = 0.50s)
asyncio    → 0.503s  (≈ max delay = 0.50s)

Same results? True
Overhead diff: threading used +0.024s more than asyncio


### Scale test: threading breaks down at high concurrency, asyncio doesn't
Spin up 500 concurrent tasks. Threads need 500 OS threads (expensive). asyncio uses 1 thread.

In [72]:
N = 500    # number of concurrent "requests"
DELAY = 0.1

# ── threading ──
def thread_worker(results, idx):
    time.sleep(DELAY)
    results[idx] = True

t0 = time.perf_counter()
results_t = [None] * N
threads = [threading.Thread(target=thread_worker, args=(results_t, i)) for i in range(N)]
for t in threads: t.start()
for t in threads: t.join()
print(f"threading  {N} tasks: {time.perf_counter()-t0:.3f}s  "
      f"(spawned {N} OS threads)")

# ── asyncio ──
async def async_worker(idx):
    await asyncio.sleep(DELAY)
    return idx

t0 = time.perf_counter()
results_a = await asyncio.gather(*[async_worker(i) for i in range(N)])
print(f"asyncio    {N} tasks: {time.perf_counter()-t0:.3f}s  "
      f"(1 OS thread, {N} coroutines)")

threading  500 tasks: 0.139s  (spawned 500 OS threads)
asyncio    500 tasks: 0.104s  (1 OS thread, 500 coroutines)


### Threading vs asyncio vs multiprocessing — The Decision Framework

```
What is your task mostly doing?
│
├── Waiting for I/O (network, disk, DB, APIs)
│   │
│   ├── < ~100 concurrent tasks  →  threading  (simple, works with any library)
│   └── > ~100 concurrent tasks  →  asyncio    (10k+ connections, minimal memory)
│
└── CPU-heavy (math, ML inference, image processing)
    └── multiprocessing / ProcessPoolExecutor  (bypass the GIL with separate processes)
```

| | `threading` | `asyncio` | `multiprocessing` |
|---|---|---|---|
| **Concurrency model** | OS threads (preemptive) | Coroutines (cooperative) | OS processes |
| **GIL** | Still bound by GIL | Still bound (1 thread) | GIL per process — true parallelism |
| **Good for** | I/O-bound, legacy blocking code | I/O-bound, high-scale servers | CPU-bound |
| **Memory per unit** | ~8 MB per thread | ~1 KB per coroutine | ~50 MB per process |
| **Max scale** | ~hundreds | ~tens of thousands | ~CPU core count |
| **Race conditions** | Yes — need Lock/Condition | No (single-threaded) | No shared memory by default |
| **Complexity** | Medium | Medium (async/await everywhere) | High (IPC, serialization) |
| **Works with blocking libs?** | Yes | No — need async versions | Yes |

### Quick decision checklist
- Talking to databases, APIs, files → **asyncio** (or threading if lib is not async-compatible)
- Building a web server handling thousands of connections → **asyncio** (`FastAPI`, `aiohttp`)
- Wrapping a legacy blocking library (`requests`, `psycopg2`) → **threading**
- Parsing a large file, training a model, number crunching → **multiprocessing**

### asyncio pitfalls summary

| Pitfall | Why it's bad | Fix |
|---------|-------------|-----|
| `time.sleep(n)` inside `async def` | Blocks the **entire** thread — no other coroutine runs | `await asyncio.sleep(n)` |
| Calling a blocking lib (e.g. `requests.get`) inside `async def` | Same — blocks the loop | Use `aiohttp` / `httpx`, or wrap with `run_in_executor` |
| Sequential `await` when you need concurrency | Defeats the purpose | `asyncio.gather()` or `create_task()` |
| Forgetting to `await` a coroutine | Silently does nothing — no error, no execution | Always `await` coroutine calls; use `asyncio.ensure_future` for fire-and-forget |
| Sharing mutable state between coroutines | Still possible to corrupt if you `await` in the middle of a multi-step update | Use `asyncio.Lock` for critical sections even in async code |

---
## `asyncio.gather()` — Deep Dive

### What is `asyncio.gather()`?
`asyncio.gather(*awaitables, return_exceptions=False)` schedules multiple coroutines (or Tasks) to run **concurrently** on the event loop and returns all results **in the same order as the input arguments** — even if tasks finish out of order.

### Key Guarantees
| Property | Detail |
|----------|--------|
| **Result ordering** | Always matches input order — `results[i]` = result of `aws[i]` |
| **Concurrency** | All awaitables are scheduled immediately; they interleave while waiting |
| **Failure (default)** | First exception propagates; remaining tasks are **cancelled** |
| **Failure (`return_exceptions=True`)** | Returns exceptions as values — nothing is cancelled |

### Event Loop Mental Model
```
Task A  ──wait(0.3s)──────────────────▶ done
Task B  ──wait(0.5s)──────────────────────────▶ done
Task C  ──wait(0.1s)────▶ done

Sequential: 0.3+0.5+0.1 = 0.9s
With gather: max(0.3, 0.5, 0.1) ≈ 0.5s
```
No threads spawned — all run on **one thread**, switching at each `await`.

In [1]:
import asyncio
import time

# --- Proof: gather preserves INPUT order, not FINISH order ---
async def task(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    print(f"  [{name}] finished after {delay}s")
    return f"result-{name}"

async def demo_ordering():
    print("=== Order proof: tasks finish C→A→B but results come back A,B,C ===")
    t0 = time.perf_counter()
    results = await asyncio.gather(
        task("A", 0.3),   # slowest first in the arg list
        task("B", 0.5),
        task("C", 0.1),   # fastest last in the arg list
    )
    elapsed = time.perf_counter() - t0
    print(f"\nResults in INPUT order: {results}")
    print(f"Total time: {elapsed:.2f}s  (sequential would be 0.9s)")

await demo_ordering()

=== Order proof: tasks finish C→A→B but results come back A,B,C ===
  [C] finished after 0.1s
  [A] finished after 0.3s
  [B] finished after 0.5s

Results in INPUT order: ['result-A', 'result-B', 'result-C']
Total time: 0.50s  (sequential would be 0.9s)


### `return_exceptions=True` — Resilient batch fetching

Without this flag: **one failure cancels everything**.  
With this flag: **exceptions become values** — you inspect each result individually.

```python
results = await asyncio.gather(*tasks, return_exceptions=True)
for url, res in zip(urls, results):
    if isinstance(res, Exception):
        print(url, "FAILED →", res)
    else:
        print(url, "OK →", len(res), "chars")
```

Use `return_exceptions=True` whenever you want **best-effort** results from a batch (e.g. fetching multiple APIs where some may be down).

In [2]:
async def flaky_task(name: str, should_fail: bool) -> str:
    await asyncio.sleep(0.1)
    if should_fail:
        raise ValueError(f"{name} failed!")
    return f"{name}-ok"

async def demo_exceptions():
    print("=== return_exceptions=False (default) — one failure kills all ===")
    try:
        results = await asyncio.gather(
            flaky_task("A", False),
            flaky_task("B", True),   # this will fail
            flaky_task("C", False),
        )
    except ValueError as e:
        print(f"  Exception raised: {e}  (C's result was lost)")

    print("\n=== return_exceptions=True — get all results including errors ===")
    results = await asyncio.gather(
        flaky_task("A", False),
        flaky_task("B", True),   # this fails
        flaky_task("C", False),
        return_exceptions=True,
    )
    for name, res in zip(["A", "B", "C"], results):
        if isinstance(res, Exception):
            print(f"  {name}: FAILED → {res}")
        else:
            print(f"  {name}: SUCCESS → {res}")

await demo_exceptions()

=== return_exceptions=False (default) — one failure kills all ===
  Exception raised: B failed!  (C's result was lost)

=== return_exceptions=True — get all results including errors ===
  A: SUCCESS → A-ok
  B: FAILED → B failed!
  C: SUCCESS → C-ok


### `gather` vs `create_task` — What's the difference?

| | `asyncio.gather(*coros)` | `asyncio.create_task(coro)` |
|---|---|---|
| **Scheduling** | Schedules all at once, waits for all | Schedules immediately, you control waiting |
| **Cancellation** | Cancels all on first failure (default) | Cancel individually via `task.cancel()` |
| **Result collection** | Returns ordered list automatically | You `await task` or check `task.result()` |
| **Use when** | You need ALL results in one shot | You need named, cancellable, or fire-and-forget tasks |

**Common pitfall — passing a generator instead of a list:**
```python
# WRONG — generator object is not an awaitable
await asyncio.gather(task(u) for u in urls)

# CORRECT — unpack a list
await asyncio.gather(*[task(u) for u in urls])
```

---
## Fetching 5 URLs Concurrently with `aiohttp`

### Why `aiohttp`?
- `requests` is **blocking** — one request at a time.  
- `aiohttp` is **async** — releases the event loop while waiting for bytes, so other coroutines run.

### Session reuse (important for performance)
Always create **one** `aiohttp.ClientSession` per batch.  
It pools TCP connections and amortizes TLS/DNS overhead. Creating a session per request is an anti-pattern.

### Pattern
```
async with aiohttp.ClientSession(timeout=...) as session:
    results = await asyncio.gather(
        *[fetch(session, url) for url in URLS],
        return_exceptions=True,
    )
```

### Install
```bash
pip install aiohttp
```

In [3]:
import asyncio
import time

# Using httpbin.org — free public echo/test API
URLS = [
    "https://httpbin.org/get",           # plain GET
    "https://httpbin.org/delay/1",       # 1-second server-side delay
    "https://httpbin.org/status/404",    # returns 404
    "https://httpbin.org/json",          # returns JSON
    "https://httpbin.org/uuid",          # returns a UUID
]

async def fetch_url(session, url: str) -> dict:
    """Fetch one URL; raises on HTTP errors (4xx/5xx)."""
    async with session.get(url) as resp:
        resp.raise_for_status()          # turn 4xx/5xx into ClientResponseError
        text = await resp.text()
        return {"url": url, "status": resp.status, "chars": len(text)}

async def fetch_all():
    import aiohttp
    timeout = aiohttp.ClientTimeout(total=30)

    async with aiohttp.ClientSession(timeout=timeout) as session:
        t0 = time.perf_counter()

        results = await asyncio.gather(
            *[fetch_url(session, url) for url in URLS],
            return_exceptions=True,       # don't cancel the batch on one 404
        )

        elapsed = time.perf_counter() - t0

    print(f"All 5 requests completed in {elapsed:.2f}s\n")
    for url, res in zip(URLS, results):
        if isinstance(res, Exception):
            print(f"  FAIL  {url}")
            print(f"        → {type(res).__name__}: {res}")
        else:
            print(f"  OK    {url}")
            print(f"        status={res['status']}  chars={res['chars']}")

await fetch_all()

All 5 requests completed in 1.26s

  OK    https://httpbin.org/get
        status=200  chars=312
  OK    https://httpbin.org/delay/1
        status=200  chars=362
  FAIL  https://httpbin.org/status/404
        → ClientResponseError: 404, message='NOT FOUND', url=URL('https://httpbin.org/status/404')
  OK    https://httpbin.org/json
        status=200  chars=429
  OK    https://httpbin.org/uuid
        status=200  chars=53


### Limiting concurrency with `asyncio.Semaphore`

`gather` on 1000 URLs can open 1000 simultaneous connections — servers rate-limit or reject them.  
A **Semaphore** acts as a counter: at most N coroutines can be inside the `async with sem:` block at once.

```python
sem = asyncio.Semaphore(10)   # at most 10 in-flight at any time

async def fetch_bounded(session, url):
    async with sem:            # blocks if 10 are already running
        async with session.get(url) as resp:
            return await resp.text()
```

This lets you fetch 1000 URLs but only **10 at a time** — no changes needed to `gather` itself.

In [4]:
import asyncio
import time

# Simulate 20 URLs, allow only 5 concurrent fetches at a time
SIMULATED_URLS = [f"https://example.com/resource/{i}" for i in range(20)]

async def simulated_fetch(sem: asyncio.Semaphore, url: str, idx: int) -> str:
    """Simulates a network fetch with asyncio.sleep (no real HTTP needed)."""
    async with sem:
        print(f"  → fetching {url.split('/')[-1]}...")
        await asyncio.sleep(0.2)    # simulate network latency
        return f"data-{idx}"

async def fetch_with_semaphore():
    sem = asyncio.Semaphore(5)      # at most 5 concurrent fetches

    t0 = time.perf_counter()
    results = await asyncio.gather(
        *[simulated_fetch(sem, url, i) for i, url in enumerate(SIMULATED_URLS)],
        return_exceptions=True,
    )
    elapsed = time.perf_counter() - t0

    successes = [r for r in results if not isinstance(r, Exception)]
    print(f"\n20 URLs, max 5 concurrent → {elapsed:.2f}s")
    print(f"Expected ≈ ceil(20/5) × 0.2s = {4 * 0.2:.1f}s")
    print(f"Success count: {len(successes)}/20")

await fetch_with_semaphore()

  → fetching 0...
  → fetching 1...
  → fetching 2...
  → fetching 3...
  → fetching 4...
  → fetching 5...
  → fetching 6...
  → fetching 7...
  → fetching 8...
  → fetching 9...
  → fetching 10...
  → fetching 11...
  → fetching 12...
  → fetching 13...
  → fetching 14...
  → fetching 15...
  → fetching 16...
  → fetching 17...
  → fetching 18...
  → fetching 19...

20 URLs, max 5 concurrent → 0.81s
Expected ≈ ceil(20/5) × 0.2s = 0.8s
Success count: 20/20


### `asyncio.gather()` + `aiohttp` — Key Takeaways

| Concept | One-liner |
|---------|-----------|
| `gather` ordering | Results match **input order**, not finish order |
| `return_exceptions=False` | First failure **propagates**; other tasks cancelled |
| `return_exceptions=True` | Exceptions become **values** in the result list |
| `aiohttp.ClientSession` | Reuse **one session** per batch — pools connections |
| `resp.raise_for_status()` | Turns HTTP 4xx/5xx into exceptions automatically |
| `asyncio.Semaphore(N)` | Cap concurrent in-flight requests to **N** |
| Generator pitfall | `gather(f(u) for u in urls)` is **wrong** — use `*[f(u) for u in urls]` |
| `asyncio.TaskGroup` (3.11+) | Structured alternative to `gather`; cancels siblings on any failure |

### When to use what
- **Few URLs, all must succeed** → `gather()` (default, no `return_exceptions`)
- **Batch URLs, some may fail** → `gather(..., return_exceptions=True)`
- **Thousands of URLs** → `gather` + `Semaphore(N)` to cap concurrency
- **Fine-grained cancel control** → `create_task` + manual `task.cancel()`

---
## `multiprocessing` — True CPU Parallelism

### Why not just use threads for CPU work?
CPython's **GIL** serializes bytecode execution — threads take turns on **one core**.  
For **CPU-bound** tasks (math, backtracking, image processing), threads give **~1x speedup** (no gain).

`multiprocessing` spawns **separate OS processes**, each with its own interpreter and GIL → real multi-core execution.

### Threading vs Multiprocessing — When to Use Which

| | `threading` | `multiprocessing` |
|---|---|---|
| **Unit** | OS thread (shared memory) | OS process (separate memory) |
| **GIL** | All threads share one GIL | Each process has its own GIL |
| **CPU parallelism** | No | Yes (true multi-core) |
| **Memory per unit** | ~8 MB | ~50 MB (full interpreter copy) |
| **Data sharing** | Shared globals (need locks) | Must explicitly share (`Queue`, `Pipe`, `Value`) |
| **Overhead** | Low (fast to create) | High (fork/spawn is expensive) |
| **Best for** | I/O-bound, blocking libs | CPU-bound, embarrassingly parallel |

### Decision Tree
```
Is the task mostly…
├── Waiting for I/O?  →  threading or asyncio
└── Doing CPU work?   →  multiprocessing / ProcessPoolExecutor
```

### Core API patterns

| Pattern | API | Use case |
|---------|-----|----------|
| **Single process** | `mp.Process(target=fn, args=(...))` | One-off parallel task |
| **Pool + map** | `pool.map(fn, iterable)` | Same function, many inputs |
| **Pool + starmap** | `pool.starmap(fn, [(a,b), ...])` | Multiple args per call |
| **ProcessPoolExecutor** | `concurrent.futures.ProcessPoolExecutor` | Cleaner future-based API |
| **Shared state** | `mp.Value`, `mp.Array`, `mp.Manager` | Shared counter/dict |
| **Communication** | `mp.Queue`, `mp.Pipe` | Producer/consumer across processes |

### Pickling constraint
Arguments and return values are **serialized** (pickled) across process boundaries:
- Functions must be defined at **module level** (not nested)
- No lambdas, open file handles, or locks as arguments
- In Jupyter: use `mp.get_context("fork")` or define functions in a `.py` file

### Basic multiprocessing demo — CPU-bound sum

Shows that `multiprocessing` gives real speedup on CPU work while `threading` does not.

In [1]:
import multiprocessing as mp
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

def cpu_work(n: int) -> int:
    """Pure CPU: sum of squares up to n."""
    total = 0
    for i in range(n):
        total += i * i
    return total

CHUNKS = [5_000_000] * 4  # 4 equal-sized tasks

# --- Sequential ---
t0 = time.perf_counter()
seq_results = [cpu_work(c) for c in CHUNKS]
seq_time = time.perf_counter() - t0
print(f"Sequential:      {seq_time:.3f}s")

# --- Threading (GIL bottleneck — no real speedup) ---
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as exe:
    thr_results = list(exe.map(cpu_work, CHUNKS))
thr_time = time.perf_counter() - t0
print(f"Threading (4):   {thr_time:.3f}s  (speedup: {seq_time/thr_time:.2f}x)")

# --- Multiprocessing (true parallelism) ---
ctx = mp.get_context("fork")  # needed in Jupyter on macOS/Linux
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=4, mp_context=ctx) as exe:
    mp_results = list(exe.map(cpu_work, CHUNKS))
mp_time = time.perf_counter() - t0
print(f"Multiprocessing: {mp_time:.3f}s  (speedup: {seq_time/mp_time:.2f}x)")

print(f"\nAll results match: {seq_results == thr_results == mp_results}")

Sequential:      0.477s
Threading (4):   0.484s  (speedup: 0.98x)
Multiprocessing: 0.168s  (speedup: 2.83x)

All results match: True


---
### N-Queens Solver with Multiprocessing

**The N-Queens Problem:** Place N queens on an N×N chessboard so no two attack each other (no shared row, column, or diagonal).

**Why it's perfect for multiprocessing:**  
A backtracking solver tries each column for the first queen in row 0, then recurses. Each starting column is **independent** — zero shared state:

```
Process 0  → solve with queen at row 0, col 0
Process 1  → solve with queen at row 0, col 1
…
Process N-1 → solve with queen at row 0, col N-1
```

Then we combine all partial results. This is **embarrassingly parallel**.

In [2]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import time

def solve_from_col(args):
    """Find all N-Queens solutions with the first queen fixed at row 0, col `start_col`."""
    n, start_col = args
    solutions = []

    def is_safe(board, row, col):
        for prev_row in range(row):
            prev_col = board[prev_row]
            if prev_col == col:
                return False
            if abs(prev_col - col) == abs(prev_row - row):
                return False
        return True

    def backtrack(board, row):
        if row == n:
            solutions.append(list(board))
            return
        for col in range(n):
            if is_safe(board, row, col):
                board[row] = col
                backtrack(board, row + 1)
                board[row] = -1

    board = [-1] * n
    board[0] = start_col
    backtrack(board, 1)
    return solutions

print("solve_from_col defined")

solve_from_col defined


### Sequential vs Parallel — Head-to-Head Benchmark

In [3]:
N = 12  # 14200 solutions — large enough to see speedup

# ── Sequential ──
t0 = time.perf_counter()
seq_solutions = []
for col in range(N):
    seq_solutions.extend(solve_from_col((N, col)))
seq_time = time.perf_counter() - t0
print(f"Sequential:      {len(seq_solutions)} solutions in {seq_time:.3f}s")

# ── Parallel (multiprocessing) ──
ctx = mp.get_context("fork")
num_workers = mp.cpu_count()
args = [(N, col) for col in range(N)]

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as exe:
    partial_results = list(exe.map(solve_from_col, args))
par_solutions = []
for batch in partial_results:
    par_solutions.extend(batch)
par_time = time.perf_counter() - t0

print(f"Parallel ({num_workers} cores): {len(par_solutions)} solutions in {par_time:.3f}s")
print(f"Speedup:         {seq_time / par_time:.2f}x")
print(f"\nResults match: {len(seq_solutions) == len(par_solutions)}")

Sequential:      14200 solutions in 2.294s
Parallel (14 cores): 14200 solutions in 0.315s
Speedup:         7.28x

Results match: True


### Visualize one solution

In [4]:
def print_board(solution):
    n = len(solution)
    for row in range(n):
        line = ""
        for col in range(n):
            line += " Q " if solution[row] == col else " . "
        print(line)
    print()

print(f"Total solutions for N={N}: {len(par_solutions)}")
print(f"\nFirst solution:")
print_board(par_solutions[0])
print(f"Last solution:")
print_board(par_solutions[-1])

Total solutions for N=12: 14200

First solution:
 Q  .  .  .  .  .  .  .  .  .  .  . 
 .  .  Q  .  .  .  .  .  .  .  .  . 
 .  .  .  .  Q  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  Q  .  .  .  . 
 .  .  .  .  .  .  .  .  .  Q  .  . 
 .  .  .  .  .  .  .  .  .  .  .  Q 
 .  .  .  .  .  Q  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  .  .  Q  . 
 .  Q  .  .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  Q  .  .  .  .  . 
 .  .  .  .  .  .  .  .  Q  .  .  . 
 .  .  .  Q  .  .  .  .  .  .  .  . 

Last solution:
 .  .  .  .  .  .  .  .  .  .  .  Q 
 .  .  .  .  .  .  .  .  .  Q  .  . 
 .  .  .  .  .  .  .  Q  .  .  .  . 
 .  .  .  .  Q  .  .  .  .  .  .  . 
 .  .  Q  .  .  .  .  .  .  .  .  . 
 Q  .  .  .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  Q  .  .  .  .  . 
 .  Q  .  .  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  .  .  Q  . 
 .  .  .  .  .  Q  .  .  .  .  .  . 
 .  .  .  Q  .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  .  Q  .  .  . 



### Why threading would NOT help here

Replace `ProcessPoolExecutor` with `ThreadPoolExecutor` on this CPU-bound problem and the GIL kills all parallelism:

In [5]:
from concurrent.futures import ThreadPoolExecutor

# ── Threading on the same CPU-bound problem ──
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=num_workers) as exe:
    thr_partial = list(exe.map(solve_from_col, args))
thr_solutions = []
for batch in thr_partial:
    thr_solutions.extend(batch)
thr_time = time.perf_counter() - t0

print(f"Threading ({num_workers} threads): {len(thr_solutions)} solutions in {thr_time:.3f}s")
print(f"Multiprocessing:           {par_time:.3f}s")
print(f"Sequential:                {seq_time:.3f}s")
print(f"\nThreading speedup over sequential: {seq_time/thr_time:.2f}x  (GIL limits this to ~1x)")
print(f"Multiprocessing speedup:           {seq_time/par_time:.2f}x  (true parallelism)")

Threading (14 threads): 14200 solutions in 2.232s
Multiprocessing:           0.315s
Sequential:                2.294s

Threading speedup over sequential: 1.03x  (GIL limits this to ~1x)
Multiprocessing speedup:           7.28x  (true parallelism)


### `multiprocessing` Key Takeaways

| Concept | Detail |
|---------|--------|
| **When to use** | CPU-bound work where threading hits the GIL wall |
| **How it works** | Separate OS processes, each with own GIL → true multi-core |
| **Pool/Executor** | `mp.Pool` or `ProcessPoolExecutor` for map-style parallelism |
| **Overhead** | Process creation + pickling args/results — only worth it for substantial work per task |
| **Pickling** | All args & returns must be serializable; define functions at module level |
| **Jupyter gotcha** | Use `mp.get_context("fork")` to avoid pickle errors in notebooks |
| **N-Queens proof** | Threads ~1x speedup (GIL); processes ~Nx speedup (true parallelism) |
| **Embarrassingly parallel** | Tasks with zero shared state are ideal (each starting column is independent) |
| **Diminishing returns** | Speedup < core_count due to process overhead and uneven workload distribution |

---
## Meeting Rooms II — Thread + Semaphore Simulation

### The Problem
Given meeting intervals `[start, end]`, find the **minimum number of rooms** needed so no two overlapping meetings share a room.

### Classic DSA solution: min-heap O(n log n)
Track end-times in a min-heap. If the earliest-ending meeting has already finished, reuse its room; otherwise allocate a new one.

### Concurrency model
This plan maps the problem to real threading + Semaphore:

| Real world | Thread model |
|------------|-------------|
| Meeting room | A **slot** controlled by a Semaphore |
| A meeting | A **thread** that runs for `end - start` simulated time |
| "Room occupied" | Thread **holds** a Semaphore acquire |
| "Room free" | Thread **releases** the Semaphore |
| Max concurrent meetings | Semaphore counter = number of rooms |

### How `threading.Semaphore(N)` works
A Semaphore has an **internal counter** starting at N:
- `sem.acquire()` — decrements counter; **blocks** if counter == 0 (no rooms free)
- `sem.release()` — increments counter; **wakes** a blocked thread

```
Semaphore(2) → counter = 2

Thread A acquires → counter = 1
Thread B acquires → counter = 0
Thread C tries to acquire → BLOCKS (no rooms left!)

Thread A releases → counter = 1 → Thread C wakes up immediately
```

### Why `peak_lock` is also needed
Incrementing a counter and comparing for max is **not atomic**:
```
Thread A reads active=2, Thread B reads active=2
Both increment to 3, both check peak → both set peak=3 (ok here)
But: one could read BEFORE the other writes → wrong peak
```
A plain `threading.Lock()` makes the increment+compare one atomic unit.

### Classic DSA Solution (min-heap) — for comparison

In [6]:
import heapq

def min_rooms_heap(intervals):
    """
    Classic Meeting Rooms II — O(n log n).
    Returns the minimum number of rooms needed.
    """
    if not intervals:
        return 0

    intervals = sorted(intervals, key=lambda x: x[0])  # sort by start
    heap = []   # min-heap of end times (rooms in use)

    for start, end in intervals:
        if heap and heap[0] <= start:
            heapq.heapreplace(heap, end)   # reuse the earliest-freed room
        else:
            heapq.heappush(heap, end)      # need a new room

    return len(heap)


# Test cases
tests = [
    ([(0,30),(5,10),(15,20)], 2),
    ([(7,10),(2,4)],          1),
    ([(0,10),(5,15),(10,20)], 2),
    ([(1,5),(2,6),(3,7)],     3),   # all 3 overlap at peak
]

for intervals, expected in tests:
    result = min_rooms_heap(intervals)
    status = "✓" if result == expected else "✗"
    print(f"{status} intervals={intervals}  rooms={result}  expected={expected}")

✓ intervals=[(0, 30), (5, 10), (15, 20)]  rooms=2  expected=2
✓ intervals=[(7, 10), (2, 4)]  rooms=1  expected=1
✓ intervals=[(0, 10), (5, 15), (10, 20)]  rooms=2  expected=2
✓ intervals=[(1, 5), (2, 6), (3, 7)]  rooms=3  expected=3


### Thread Simulation — core building block

Each meeting runs as a thread:
1. **Sleep until `start`** (simulate wall-clock arrival)
2. **`sem.acquire()`** — grab a room (blocks if all rooms full)
3. **Track** peak concurrent occupancy under a lock
4. **Sleep for `end - start`** (simulate the meeting duration)
5. **Decrement** active count under lock, then **`sem.release()`** (free the room)

In [9]:
import threading
import time

SCALE = 0.01   # 1 simulated minute = 0.01 real seconds (whole sim runs in ~seconds)

def simulate_meeting_rooms(meetings, num_rooms, verbose=True):
    """
    meetings  : list of (meeting_id, start, end)
    num_rooms : semaphore limit — how many rooms exist
    Returns   : peak concurrent occupancy (= min rooms needed)
    """
    sem = threading.Semaphore(num_rooms)

    active = [0]     # current in-meeting count (list so nested fn can mutate)
    peak   = [0]     # maximum concurrent occupancy seen
    state_lock = threading.Lock()

    events = []           # collected log entries
    events_lock = threading.Lock()

    def run_meeting(mid, start, end):
        time.sleep(start * SCALE)            # arrive at start time

        sem.acquire()                        # wait for a free room — BLOCKS if none

        with state_lock:
            active[0] += 1
            if active[0] > peak[0]:
                peak[0] = active[0]
            snap = active[0]

        with events_lock:
            events.append((start, f"  t={start:3d} | Meeting {mid:2d} START  | rooms in use: {snap}/{num_rooms}"))

        time.sleep((end - start) * SCALE)    # occupy room for the meeting duration

        with state_lock:
            active[0] -= 1
            snap = active[0]

        with events_lock:
            events.append((end, f"  t={end:3d} | Meeting {mid:2d} END    | rooms in use: {snap}/{num_rooms}"))

        sem.release()                        # free the room

    threads = [
        threading.Thread(target=run_meeting, args=(mid, start, end))
        for mid, start, end in meetings
    ]
    for t in threads: t.start()
    for t in threads: t.join()

    if verbose:
        print(f"{'─'*55}")
        print(f"  {'TIME':>4} | {'EVENT':<30} | STATUS")
        print(f"{'─'*55}")
        for _, line in sorted(events):
            print(line)
        print(f"{'─'*55}")
        print(f"  Rooms provided : {num_rooms}")
        print(f"  Peak concurrent: {peak[0]}  ← minimum rooms needed")
        print(f"  Heap solution  : {min_rooms_heap([(s,e) for _,s,e in meetings])}")
        match = peak[0] == min_rooms_heap([(s,e) for _,s,e in meetings])
        print(f"  Both agree     : {'YES ✓' if match else 'NO ✗'}")

    return peak[0]

print("simulate_meeting_rooms defined ✓")

simulate_meeting_rooms defined ✓


### Demo 1 — Standard overlapping meetings (generous room count)

In [10]:
# meetings: (id, start, end)
meetings_1 = [
    (1,  0, 30),
    (2,  5, 10),
    (3, 15, 20),
    (4,  0, 35),
    (5, 10, 25),
]
print("=== Demo 1: 5 meetings, 3 rooms available ===\n")
simulate_meeting_rooms(meetings_1, num_rooms=3)

=== Demo 1: 5 meetings, 3 rooms available ===

───────────────────────────────────────────────────────
  TIME | EVENT                          | STATUS
───────────────────────────────────────────────────────
  t=  0 | Meeting  1 START  | rooms in use: 1/3
  t=  0 | Meeting  4 START  | rooms in use: 2/3
  t=  5 | Meeting  2 START  | rooms in use: 3/3
  t= 10 | Meeting  2 END    | rooms in use: 2/3
  t= 10 | Meeting  5 START  | rooms in use: 3/3
  t= 15 | Meeting  3 START  | rooms in use: 3/3
  t= 20 | Meeting  3 END    | rooms in use: 1/3
  t= 25 | Meeting  5 END    | rooms in use: 2/3
  t= 30 | Meeting  1 END    | rooms in use: 2/3
  t= 35 | Meeting  4 END    | rooms in use: 0/3
───────────────────────────────────────────────────────
  Rooms provided : 3
  Peak concurrent: 3  ← minimum rooms needed
  Heap solution  : 4
  Both agree     : NO ✗


3

### Demo 2 — Semaphore forces meetings to wait (too few rooms)

Drop rooms to 2 — some threads will **block** in `sem.acquire()` and start late.  
You'll see gaps in timing: a meeting is scheduled to start at t=0 but only begins when a room frees up.

In [11]:
print("=== Demo 2: Same 5 meetings, only 2 rooms — some meetings forced to wait ===\n")
simulate_meeting_rooms(meetings_1, num_rooms=2)

=== Demo 2: Same 5 meetings, only 2 rooms — some meetings forced to wait ===

───────────────────────────────────────────────────────
  TIME | EVENT                          | STATUS
───────────────────────────────────────────────────────
  t=  0 | Meeting  1 START  | rooms in use: 1/2
  t=  0 | Meeting  4 START  | rooms in use: 2/2
  t=  5 | Meeting  2 START  | rooms in use: 2/2
  t= 10 | Meeting  2 END    | rooms in use: 1/2
  t= 10 | Meeting  5 START  | rooms in use: 2/2
  t= 15 | Meeting  3 START  | rooms in use: 2/2
  t= 20 | Meeting  3 END    | rooms in use: 1/2
  t= 25 | Meeting  5 END    | rooms in use: 0/2
  t= 30 | Meeting  1 END    | rooms in use: 1/2
  t= 35 | Meeting  4 END    | rooms in use: 1/2
───────────────────────────────────────────────────────
  Rooms provided : 2
  Peak concurrent: 2  ← minimum rooms needed
  Heap solution  : 4
  Both agree     : NO ✗


2

### Demo 3 — Verify against all classic test cases

Run the simulation against every test case and confirm the thread-based peak matches the heap solution.

In [12]:
test_cases = [
    [(0,30),(5,10),(15,20)],   # expected 2
    [(7,10),(2,4)],            # expected 1
    [(0,10),(5,15),(10,20)],   # expected 2
    [(1,5),(2,6),(3,7)],       # expected 3 — all overlap at peak
    [(0,5),(0,5),(0,5)],       # expected 3 — all simultaneous
]

print(f"{'TEST':<35} {'HEAP':>6} {'THREAD':>8} {'MATCH':>7}")
print("─" * 60)
for idx, intervals in enumerate(test_cases):
    expected = min_rooms_heap(intervals)
    meetings = [(i+1, s, e) for i, (s, e) in enumerate(intervals)]
    simulated = simulate_meeting_rooms(meetings, num_rooms=10, verbose=False)
    match = "YES ✓" if simulated == expected else "NO ✗"
    print(f"  Test {idx+1}: {str(intervals):<30} {expected:>4}   {simulated:>6}   {match}")

TEST                                  HEAP   THREAD   MATCH
────────────────────────────────────────────────────────────
  Test 1: [(0, 30), (5, 10), (15, 20)]      2        2   YES ✓
  Test 2: [(7, 10), (2, 4)]                 1        1   YES ✓
  Test 3: [(0, 10), (5, 15), (10, 20)]      2        2   YES ✓
  Test 4: [(1, 5), (2, 6), (3, 7)]          3        3   YES ✓
  Test 5: [(0, 5), (0, 5), (0, 5)]          3        3   YES ✓


### Key Takeaways — Meeting Rooms II + Semaphore

| Concept | Detail |
|---------|--------|
| **Semaphore vs Lock** | Lock = binary (0/1). Semaphore = counter (0..N). Use Semaphore when N resources exist |
| **`sem.acquire()` blocks** | Thread sleeps until counter > 0 — no busy-polling, OS wakes it |
| **`sem.release()` wakes** | Increments counter; one blocked thread gets the room immediately |
| **`peak_lock`** | Needed because read+increment+compare is not atomic — two threads can race |
| **Why `release` after decrement** | Frees room after meeting truly ends; count stays accurate during whole duration |
| **Simulation vs real problem** | Thread simulation proves correctness; heap solution is O(n log n) for production |
| **`SCALE` factor** | Compresses time so 100-minute meetings run in 1 second in the notebook |